In [1]:
import glob
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0,"/user/work/kr21883/acrg/")
import acrg.obs as getobs
from acrg.hbmcmc.run_hbmcmc import hbmcmc_extract_param
import acrg.name.name as name

%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
from acrg.hbmcmc.run_hbmcmc import extract_mcmc_type,define_mcmc_function

/user/home/kr21883/miniconda3/envs/acrg/lib/python3.10/site-packages/arviz/data/base.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from acrg.hbmcmc.run_hbmcmc import extract_mcmc_type,define_mcmc_function

In [4]:
def get_inversion_estimates(folder_path, run_name, run_type="MAP", spec=None, year=2016, country="BRAZIL", uncertainty="95", return_unc=False):
    # this should get renamed to get estimates!! it only gets MAP estimates if spec=MAP
    months = ["01","02", "03", "04", "05", "06", "07", "08","09", "10", "11", "12"]
    estimates = np.zeros((12,))
    uncertainties = np.zeros((12,2))
    uncertainties = np.where(uncertainties==0, np.nan, uncertainties)
    if spec is None:
        path = f"{folder_path}/CH4_*{run_name}_{year}"
    else:
        path = f"{folder_path}/CH4_*{run_name}-fps_{spec}_{year}"
    #print(path)
    for i, month in enumerate(months):
        #print(glob.glob(f"{path}-{month}-01.nc"))
        if len(glob.glob(f"{path}-{month}-01.nc")) == 1:
            try:
                d = xr.open_dataset(glob.glob(f"{path}-{month}-01.nc")[0])
                estimates[i] = d.sel(countrynames=country).countrymean.values
                if run_type!="MAP":
                    uncertainties[i,0] = d.sel(countrynames=country)[f"country{uncertainty}"].values[0]
                    uncertainties[i,1] = d.sel(countrynames=country)[f"country{uncertainty}"].values[1]
                d.close()
            except Exception as e:
                print(f"opening or extracting the data didnt work, for {month}!")
                print(f"with error {e}")
                print(f"maybe check path {path}")
        else:
            estimates[i] = np.nan
    if np.sum(np.isnan(estimates))==12:
        print("No months were loaded! check folder")
        print(path)
        
    if return_unc:        
        return estimates, uncertainties
    else:
        return estimates


def get_apriori(folder, run_name, spec=None, year=2016, uncertainty="95", country="MOROCCO"):
    months = ["01","02", "03", "04", "05", "06", "07", "08","09", "10", "11", "12"]
    apriori = np.zeros((12))
    if spec is None:
        path = f"/user/work/ef17148/acrg/satellite_outputs/NEW/SAHARA/{folder}/CH4_*{run_name}-fps_{year}"
    else:
        path = f"/user/work/ef17148/acrg/satellite_outputs/NEW/SAHARA/{folder}/CH4_*{run_name}-fps_{spec}_{year}"
    #print(path)
    for i, month in enumerate(months):
        files = glob.glob(f"{path}-{month}-01.nc")
        if len(files) > 1:
            files = [f for f in files if "-MAP-" not in f]
        
        if len(files) == 1:
            d = xr.open_dataset(glob.glob(f"{path}-{month}-01.nc")[0])
            apriori[i] = d.sel(countrynames=country).countryapriori.values
            d.close()
        else:
            apriori[i] = np.nan

    if np.sum(np.isnan(apriori))==12:
        print("No months were loaded! check folder")
        print(path)        
        
    return apriori

In [5]:
config_file = "/user/work/ef17148/acrg/acrg/hbmcmc/bc_emulation_tests/hbmcmc_input_GOSAT-brazil_bc_emulation_test.ini"

In [6]:
month_dates = {"Jan":{"start_date":"2017-01-01","end_date":"2017-02-01"},
               "Feb":{"start_date":"2017-02-01","end_date":"2017-03-01"},
               "Mar":{"start_date":"2017-03-01","end_date":"2017-04-01"},
               "Apr":{"start_date":"2017-04-01","end_date":"2017-05-01"},
               "May":{"start_date":"2017-05-01","end_date":"2017-06-01"},
               "Jun":{"start_date":"2017-06-01","end_date":"2017-07-01"},
               "Jul":{"start_date":"2017-07-01","end_date":"2017-08-01"},
               "Aug":{"start_date":"2017-08-01","end_date":"2017-09-01"},
               "Sep":{"start_date":"2017-09-01","end_date":"2017-10-01"},
               "Oct":{"start_date":"2017-10-01","end_date":"2017-11-01"},
               "Nov":{"start_date":"2017-11-01","end_date":"2017-12-01"},
               "Dec":{"start_date":"2017-12-01","end_date":"2017-12-31"}}

In [7]:
month="Feb"

mcmc_type = extract_mcmc_type(config_file)
mcmc_function = define_mcmc_function(mcmc_type)
print(f"Using MCMC type: {mcmc_type} - function {mcmc_function.__name__}(...)")

param = hbmcmc_extract_param(config_file,mcmc_type, start_date=month_dates[month]["start_date"], end_date=month_dates[month]["end_date"])

PermissionError: [Errno 13] Permission denied: '/user/work/ef17148/acrg/acrg/hbmcmc/bc_emulation_tests/hbmcmc_input_GOSAT-brazil_bc_emulation_test.ini'

In [ ]:
param["outputname"] = "GOSAT-BRAZIL-LAND-TEST"

In [ ]:
mcmc_function(**param)

In [ ]:
months_to_run = ["Jan", "Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
for month in months_to_run:
    print(f"Running inversion for month: {month}")
    param = hbmcmc_extract_param(config_file,mcmc_type, start_date=month_dates[month]["start_date"], end_date=month_dates[month]["end_date"])
    mcmc_results = mcmc_function(**param)

## loading and plotting

In [ ]:
load_runames = {"default":"default-test", "true subsampled":"true_bcs_subsampled", "pred subsampled":"pred_bcs_subsampled", "pred full":"GOSAT-BRAZIL-pred_bcs_full", "true full":"GOSAT-BRAZIL-true_bcs_full", "LAND pred full":"GOSAT-BRAZIL-LAND-pred_bcs_full", "LAND true full":"GOSAT-BRAZIL-LAND-true_bcs_full"}

In [ ]:
estimates = {}
for runame in load_runames.keys():
    print(f"Loading inversion results for: {runame}")
    estimates_here = get_inversion_estimates("/user/work/ef17148/acrg/satellite_outputs/bc_inversions/first_MAP_experiments/", run_name=load_runames[runame], run_type="MAP", spec=None, year=2017, country="BRAZIL")
    estimates_here[estimates_here>1e17] = np.nan
    estimates[runame] = estimates_here/1e12

In [ ]:
full_estimates = {}
load_runames = {"pred full":"GOSAT-BRAZIL-pred_bcs", "true full":"GOSAT-BRAZIL-true_bcs", "LAND pred full":"GOSAT-BRAZIL-LAND-pred_bcs", "LAND true full":"GOSAT-BRAZIL-LAND-true_bcs"}
for runame in load_runames.keys():
    print(f"Loading inversion results for: {runame}")
    estimates_here, uncertainty = get_inversion_estimates("/user/work/ef17148/acrg/satellite_outputs/bc_inversions/first_inversion/", run_name=load_runames[runame], run_type="MCMC", spec=None, year=2017, country="BRAZIL", return_unc=True)
    estimates_here[estimates_here>1e17] = np.nan
    full_estimates[runame] = (estimates_here/1e12, uncertainty/1e12)

In [ ]:
br = xr.open_dataset("/user/work/ef17148/acrg/satellite_outputs/bc_inversions/first_MAP_experiments/CH4_SOUTHAMERICA_GOSAT-BRAZIL-LAND-true_bcs_full_2017-06-01.nc").sel(countrynames="BRAZIL")

In [ ]:
def plot_estimates(dict_estimates, dict_full_estimates, est_to_plot, est_to_plot_full):
    fig, ax = plt.subplots(1,1, figsize=(8,5))
 
    for estimate_label in est_to_plot:
        p = ax.plot(np.arange(1,13), dict_estimates[estimate_label], label=estimate_label)
        ax.scatter(np.arange(1,13), dict_estimates[estimate_label], color=p[0].get_color())

    for estimate_label in est_to_plot_full:
        p = ax.plot(np.arange(1,13), dict_full_estimates[estimate_label][0], label=estimate_label)
        ax.scatter(np.arange(1,13), dict_full_estimates[estimate_label][0], color=p[0].get_color())
        # make fill_between the same color as plot
        ax.fill_between(np.arange(1,13), dict_full_estimates[estimate_label][1][:,0], dict_full_estimates[estimate_label][1][:,1], alpha=0.3, color=p[0].get_color())
    
    ax.set_xlabel("Month")
    ax.set_ylabel("Inversion Estimate (Tg CH4/month)")
    ax.set_title("Brazil Inversion Estimates 2017")
    ax.legend()
    plt.show()

In [ ]:
plot_estimates(estimates, {}, ["default", "LAND true full", "LAND pred full"], [])

In [ ]:
plot_estimates(estimates, full_estimates, [], ["pred full", "true full", "LAND pred full", "LAND true full"])

In [ ]:
plot_estimates(estimates, full_estimates, [], ["pred full", "true full"])

In [ ]:
plot_estimates(estimates, full_estimates, [], ["LAND pred full", "LAND true full"])

In [ ]:
plot_estimates(estimates, full_estimates, [], ["true full", "LAND true full"])